# Distance Contraction — Ion-Migration Pathway Screener

**Author:** Ramanuja Srinivasan Saravanan (University of Maryland)  
**Supervisor:** Prof. Yifei Mo  
**Version:** 1.0  
**License:** MIT  

---

## Overview

This notebook screens candidate ion-migration pathways in solid-state electrolyte frameworks
using the *distance-contraction* descriptor.

**The core idea in plain language:**  
When a mobile ion (Na⁺, Li⁺, K⁺, Mg²⁺, …) hops between two lattice sites, it
has to squeeze past neighbouring cations and anions. At the tightest point —
the transition state (TS) — those neighbours are *closer* than their equilibrium
bond lengths. We quantify that squeeze as a **distance contraction**: how much
shorter each bond is at the TS compared with a reference distance in the pristine
structure. A large contraction means a high energy barrier; a small contraction
means the path is relatively open and the ion can hop easily.

**Technically:**  
All mobile-ion site pairs within a user-defined hop-distance cutoff are first
enumerated. Crystal symmetry is then used to reduce these to a set of
*symmetry-unique* paths — pairs that are crystallographically inequivalent and
therefore have distinct energy barriers. The computed descriptor values are
subsequently mapped back onto *all* original pairs via the symmetry mapping,
so that the full percolation network (which requires every path, not just the
unique ones) can be reconstructed correctly for dimensionality analysis.

For each symmetry-unique path the notebook:
1. Generates linearly interpolated NEB images between the start and end sites.
2. Displaces the migrating ion transversely to minimise worst-case steric clash
   with neighbouring ions, producing a more physically realistic pathway geometry.
3. Evaluates four transverse configurations (cation-only neighbours vs. all
   neighbours, positive vs. negative direction) and selects the one with the
   smallest transition-state contraction.
4. Identifies the transition-state image as the one exhibiting maximum
   distance contraction relative to a reference (mainly *path*, and optionally *struct*, *stddev*).
5. Maps descriptor values back to all symmetry-equivalent paths and
   incrementally adds paths — sorted by contraction — to determine the
   dimensionality (1-D / 2-D / 3-D) at which the percolation network first
   becomes connected.

All data-loading, geometry, and pipeline code lives in **`dc_core.py`**
(imported in Section 1) — this notebook only holds setup, the interactive
controls, and the results display.

---

## Required input files

## Upload Your CIF File

In the left sidebar, click the 📁 **Files** icon → ⬆ **Upload** button, and select your CIF file. The file must be named as:

- `<ICSD_number>.cif` (e.g. `29962.cif`)

All other required data files are downloaded automatically from GitHub in Section 1 — no additional uploads needed. The `*_bond_stddev.csv` files are only downloaded if you select "Struct + stddev + path" in the Section 2 widget.

| File | Description |
|---|---|
| `<ICSD>.cif` or `<ICSD>_ord_rama.cif` | Crystal structure (CIF format) |
| `periodic_table.json` | Shannon ionic radii & full periodic table data |
| `Na_bond_stddev.csv`, `Li_bond_stddev.csv`, ... | Per-element bond-distance statistics (only if "stddev" category is selected) |

---

## How to use

1. Run **Section 1** (install, download `dc_core.py`, and import it) once per Colab/Jupyter session.
2. Use the interactive widget in **Section 2** to choose the mobile-ion element,
   maximum hop-distance cutoff, number of NEB images, extra reference categories,
   and ICSD code.
3. Click **Run** — results are written to `<ICSD>_deviation.csv` (or inside
   `<ICSD>_neb/` if *Create folder = Yes*).
4. See **Section 3** for the results summary.


## Section 1 — Environment Setup

Install packages and load the core library (`dc_core.py`).  
Run this cell once at the start of each Colab/Jupyter session.


In [ ]:
# ── Package installation (Colab) ──────────────────────────────────────────
# Skip if packages are already installed in your local environment.
!pip install --upgrade --quiet ase
!pip install --quiet pymatgen==2025.6.14 natsort


In [ ]:
# ── Download dc_core.py + base data files, then import ────────────────────
import os, sys, subprocess
import pandas as pd  # used directly by the Section 3 result-display cells

os.chdir("/content")

_MODULE_URL = "https://raw.githubusercontent.com/Rama-MSE-UMD/Distance-contraction-descriptor/main/dc_core.py"

if not os.path.exists("dc_core.py"):
    subprocess.run(["wget", "-q", _MODULE_URL])
if not os.path.exists("dc_core.py"):
    raise FileNotFoundError(
        "Could not fetch dc_core.py automatically.\n"
        "Either add it to the GitHub repo at:\n  " + _MODULE_URL + "\n"
        "or upload dc_core.py manually to /content (Colab: Files pane → upload), then re-run this cell."
    )

sys.path.insert(0, "/content")
import dc_core as dc

# periodic_table.json + CIF files — always needed, independent of the
# Section 2 "Extra categories" choice. The stddev CSVs are NOT downloaded
# here; they're fetched lazily inside dc.run_pipeline(...) only if you pick
# "Struct + stddev + path" in Section 2.
dc.download_base_files()
print("dc_core loaded.")


## Section 2 — Interactive Workflow

Use the widget below to configure and launch the analysis.

| Parameter | Meaning |
|---|---|
| **Element** | Mobile ion species |
| **Max path length (Å)** | Maximum hop distance considered |
| **# of Images** | Number of NEB interpolation images (excluding end-points) |
| **Create folder** | Write per-path CIF files to `<ICSD>_neb/` |
| **Extra categories** | Whether to also compute the `struct` and `stddev` reference categories (in addition to `path`). Choosing "Struct + stddev + path" is what triggers the one-time download of the per-element `*_bond_stddev.csv` files — nothing is downloaded up front. |
| **ICSD number** | Numeric identifier; expects `<ICSD>_ord_rama.cif` or `<ICSD>.cif` |

**Output files:**

| File | Contents |
|---|---|
| `<ICSD>_data.csv` | Per-path distance-contraction metrics |
| `<ICSD>_deviation.csv` | Per-structure dimensionality + contraction summary |
| `<ICSD>_uniquepair.csv` | All pairs with symmetry-mapped descriptor values |
| `<ICSD>_uniquesymm.csv` | Symmetry-unique paths only |
| `neb_<N>_linear.cif` | Linear (unoptimised) NEB path CIF |
| `neb_<N>_predicted_<mode>_<dir>.cif` | Transverse-optimised path CIFs |
| `image_0<N>.cif` | Individual NEB image structures |
| `neb_<N>_merged.cif` | All images of path *N* merged into one structure |
| `*Dim_struct_path.cif` | Mobile-ion network at each dimensionality threshold |


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Widget definitions ────────────────────────────────────────────────────
element_selector       = widgets.ToggleButtons(
    options=["Na", "Li", "Mg", "K", "Ca", "Al", "Zn"], value="Na", description="Element:")
max_path_selector      = widgets.ToggleButtons(
    options=[3, 4, 5, 6, 7], value=7, description="Max path length (Å):")
nimages_selector       = widgets.ToggleButtons(
    options=[3, 5, 7, 9, 11, 13], value=5, description="# of Images:")
create_folder_selector = widgets.ToggleButtons(
    options=["Yes", "No"], value="Yes", description="Create folder:")
# Deciding this to "Struct + stddev + path" is what triggers dc.run_pipeline
# to download the *_bond_stddev.csv files (see dc_core.ensure_stddev_tables) —
# there's no separate pre-set flag; the widget IS the decision.
categories_selector = widgets.ToggleButtons(
    options=["Struct + stddev + path", "Path only"],
    value="Path only",
    description="Extra categories:")
icsd_input = widgets.Text(
    value="", description="ICSD number:", placeholder="e.g., 123456")
run_btn = widgets.Button(description="Run", button_style="primary")
out     = widgets.Output()
ui = widgets.VBox([
    element_selector, max_path_selector, nimages_selector,
    create_folder_selector, categories_selector, icsd_input, run_btn, out
])

# ════════════════════════════════════════════════════════════════════════════
# Widget callback
# ════════════════════════════════════════════════════════════════════════════
_UI_READY = {"value": False}

def on_run_clicked(_):
    if not _UI_READY["value"]:
        return
    with out:
        clear_output(wait=True)
        st_name = icsd_input.value.strip()
        if not st_name.isdigit():
            print("ERROR: Please enter a valid numeric ICSD code.")
            return
        ele = element_selector.value
        oxi = dc.OXI_STATE.get(ele)
        if oxi is None:
            print(f"ERROR: Oxidation state not defined for '{ele}'.")
            return
        compute_extra = (categories_selector.value == "Struct + stddev + path")
        print("Settings")
        print(f"  Element          : {ele}")
        print(f"  Oxidation state  : {oxi}")
        print(f"  Max path length  : {max_path_selector.value} Å")
        print(f"  NEB images       : {nimages_selector.value}")
        print(f"  Create folder    : {create_folder_selector.value}")
        print(f"  Extra categories : {categories_selector.value}")
        print(f"  ICSD             : {st_name}\n")

        global df_out, low_energy_df, df1, dim_struct, dim_path, dim_stddev, _st_name, _structure
        df_out, low_energy_df, df1, dim_struct, dim_path, dim_stddev, _st_name, _structure = dc.run_pipeline(
            element_symbol=ele,
            element_oxi=oxi,
            st_name=st_name,
            max_path_length=max_path_selector.value,
            nimages=nimages_selector.value,
            create_folder=create_folder_selector.value,
            compute_extra=compute_extra,
        )

run_btn.on_click(on_run_clicked)
display(ui)
_UI_READY["value"] = True


## Section 3 — Results Summary

Run the cells below after clicking **Run** in Section 2.

- **Cell A**: 3 × 6 summary — contraction threshold (Å) at which each percolation dimensionality is first reached, for each reference scheme
- **Cell B**: Per-path contraction table (all paths, mapped via symmetry)
- **Cell C**: Unique NEB folders, clustered by contraction-value similarity


In [ ]:
# ── 3 × 6 Descriptor summary table ───────────────────────────────────────
# Rows    : reference scheme (struct, path, stddev)
# Columns : dimensionality thresholds (0-D, 1-D, 2-D, 3-D), Struct, Composition
#
# Each value is the distance-contraction (Å) at which that dimensionality
# is first reached when paths are added in order of increasing contraction.
# None means that dimensionality was not achieved by any path combination.

def _get_dim(d, dim):
    val = d.get(dim, None)
    if val is None:
        return None
    try:
        return round(float(val), 2)
    except (TypeError, ValueError):
        return val

rows = []
for label, dim_dict in [
    ("struct", dim_struct),
    ("path",   dim_path),
    ("stddev", dim_stddev),
]:
    rows.append({
        "Reference":   label,
        "0-D":         _get_dim(dim_dict, 0),
        "1-D":         _get_dim(dim_dict, 1),
        "2-D":         _get_dim(dim_dict, 2),
        "3-D":         _get_dim(dim_dict, 3),
        "Struct":      _st_name,
        "Composition": str(_structure.composition),
    })

summary_df = pd.DataFrame(rows).set_index("Reference")
print("Distance contraction descriptor summary (Å):")
display(summary_df)


In [ ]:
# ── Unique NEB folders (clustered by contraction-value similarity) ───────
unique_folders, cluster_map = dc.find_unique_neb_paths(
    df1, value_col="Max_contraction_path", tolerance=0.02)

print(f"{len(unique_folders)} unique paths out of {len(df1)} total (tol=0.02 Å):")
print(unique_folders)


In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

icsd_number = icsd_input.value.strip()

df_results = pd.read_csv(f"{icsd_number}_neb/{icsd_number}_data.csv")

# Round all numeric columns to 2 decimal places
numeric_cols = df_results.select_dtypes(include="number").columns
df_results[numeric_cols] = df_results[numeric_cols].round(2)

overview_cols = [
    "NEB", "NEB1",
    "Category_stddev", "Max_contraction_stddev",
    "Category_path",   "Max_contraction_path",
]
display(df_results[[c for c in overview_cols if c in df_results.columns]])


In [ ]:
print(os.getcwd()); print([i for i in os.listdir() if i.endswith("neb")])


In [ ]:
# rm -r <ICSD>_neb   # uncomment + fill in to delete a run's output folder


In [ ]:
os.chdir("/content")  # reset working directory back to /content
